<div style='background:linear-gradient(135deg,#08233E 0%,#005BAC 100%);padding:48px 40px;border-radius:20px;margin-bottom:32px'>
  <h1 style='color:#fff;font-size:36px;margin:0 0 8px'>Building a Restaurant Food-Ordering Chatbot</h1>
  <h2 style='color:#08BCEF;font-size:24px;margin:0 0 20px;font-weight:400'>using LLM and RAG</h2>
  <p style='color:#CFEAF5;font-size:16px;margin:0 0 4px'><b>CMC Restaurant — QR AI Ordering System</b></p>
  <p style='color:#CFEAF5;font-size:14px;margin:0'>Học phần: Học máy và Khai phá dữ liệu</p>
</div>

<div style='background:#EAF7FC;border:1px solid #CFEAF5;border-radius:14px;padding:24px 28px;margin:8px 0 24px'>
  <h3 style='color:#08233E;margin:0 0 10px'>Tóm tắt (Abstract)</h3>
  <p style='color:#08233E;font-size:14px;line-height:1.8;margin:0 0 12px'>
  Đồ án xây dựng <b>chatbot tư vấn gọi món nhà hàng</b> cho hệ thống đặt món qua mã QR, kết hợp
  <b>mô hình ngôn ngữ lớn (LLM)</b> với kỹ thuật <b>Retrieval-Augmented Generation (RAG)</b>.
  Hệ thống truy xuất tri thức bằng <b>BM25</b> (mô hình xếp hạng trong Truy hồi thông tin), đưa ngữ
  cảnh đã trích vào prompt, rồi sinh câu trả lời có kiểm soát qua <b>guardrails</b> và
  <b>output parser</b>. Tri thức nền tích hợp hai kỹ thuật khai phá dữ liệu: <b>luật kết hợp</b>
  (cặp món hay đi cùng) và <b>gợi ý theo nội dung</b> (đối chiếu khẩu vị ↔ tag món). Hệ thống được
  đánh giá định lượng trên tập 15 câu hỏi vàng theo ba chỉ số: Retrieval Hit Rate@5, Guardrail
  Accuracy và Overall Pass Rate.
  </p>
  <h4 style='color:#005BAC;margin:8px 0 6px'>Mục tiêu</h4>
  <ol style='color:#08233E;font-size:14px;line-height:1.9;margin:0;padding-left:20px'>
    <li>Phân tích bài toán tư vấn gọi món và đặc thù dữ liệu tiếng Việt, miền hẹp, ít nhãn.</li>
    <li>Lựa chọn & lập luận phương pháp truy hồi (BM25) và sinh có grounding (RAG) phù hợp ràng buộc.</li>
    <li>Ứng dụng kỹ thuật khai phá dữ liệu (luật kết hợp, gợi ý theo nội dung) vào tri thức nền.</li>
    <li>Thiết kế cơ chế an toàn chống bịa món/giá và chống AI tự đặt đơn.</li>
    <li>Đánh giá định lượng và phân tích hạn chế của hệ thống.</li>
  </ol>
</div>

<div style='background:#F2FAFD;border-left:4px solid #005BAC;padding:20px 24px;border-radius:0 12px 12px 0;margin-bottom:24px'>
  <h3 style='color:#08233E;margin:0 0 12px'>Mục lục</h3>
  <ol style='color:#08233E;font-size:15px;line-height:2'>
    <li><a href='#sec1' style='color:#005BAC'>Tổng quan & Kiến trúc hệ thống</a></li>
    <li><a href='#sec2' style='color:#005BAC'>Knowledge Base — Kho tri thức nhà hàng</a></li>
    <li><a href='#sec3' style='color:#005BAC'>BM25 Retriever — Truy xuất thông tin</a></li>
    <li><a href='#sec4' style='color:#005BAC'>Prompt Engineering — Xây dựng ngữ cảnh cho LLM</a></li>
    <li><a href='#sec5' style='color:#005BAC'>Guardrails — Hệ thống an toàn AI</a></li>
    <li><a href='#sec6' style='color:#005BAC'>Output Parser — Xử lý & Kiểm duyệt đầu ra</a></li>
    <li><a href='#sec7' style='color:#005BAC'>Evaluation — Đánh giá hiệu năng RAG</a></li>
    <li><a href='#sec8' style='color:#005BAC'>End-to-End Demo — Gọi LLM thật</a></li>
    <li><a href='#sec9' style='color:#005BAC'>Kết luận & Hướng phát triển</a></li>
    <li><a href='#refs' style='color:#005BAC'>Tài liệu tham khảo</a></li>
  </ol>
</div>

In [1]:
# ══════════════════════════════════════════════════
# SETUP — Helpers cho trình bày
# ══════════════════════════════════════════════════
from IPython.display import HTML, display

def info_card(title, value, detail="", color="#005BAC"):
    return f"""
    <div style='display:inline-block;min-width:180px;border:1px solid #CFEAF5;border-radius:14px;
                padding:16px 20px;margin:0 10px 10px 0;background:#fff'>
      <div style='color:#667A91;font-size:11px;font-weight:700;text-transform:uppercase;letter-spacing:0.06em'>{title}</div>
      <div style='color:{color};font-size:28px;font-weight:900;margin:6px 0 2px'>{value}</div>
      <div style='color:#667A91;font-size:12px'>{detail}</div>
    </div>"""

print("Setup complete.")

Setup complete.


<a id='sec1'></a>
<div style='border-bottom:3px solid #005BAC;padding-bottom:8px;margin:40px 0 20px'>
  <h2 style='color:#08233E;margin:0'>1. Tổng quan & Kiến trúc hệ thống</h2>
</div>

### 1.1 Bài toán

Xây dựng chatbot **hỗ trợ gọi món** trong nhà hàng, kết hợp:
- **LLM** (Large Language Model) để hiểu ngôn ngữ tự nhiên và tạo câu trả lời
- **RAG** (Retrieval-Augmented Generation) để bổ sung context từ knowledge base trước khi sinh câu trả lời

### 1.2 Yêu cầu đặc biệt

| Yêu cầu | Giải pháp |
|:---------|:----------|
| AI **không được tự đặt đơn** | Guardrails + `requires_customer_confirmation: true` |
| AI **không được bịa giá** | Output parser validate menu_item_id |
| AI **không được bịa món** | Cross-check với database menu thật |
| Nếu LLM **offline** vẫn hoạt động | Fallback answer từ RAG context |

### 1.3 Phân tích & lựa chọn phương pháp (góc nhìn Học máy & Khai phá dữ liệu)

Trước khi triển khai, ta phân tích đặc thù bài toán để chọn phương pháp phù hợp — không chọn theo cảm tính.

**Đặc thù dữ liệu & ràng buộc:**
- Ngôn ngữ **tiếng Việt có dấu**, miền **hẹp** (một nhà hàng), **không có tập dữ liệu gán nhãn** lớn để huấn luyện.
- Tri thức (menu, chính sách, combo) **thay đổi thường xuyên** → cần cập nhật không qua huấn luyện lại.
- Yêu cầu **an toàn cao**: tuyệt đối không bịa món/giá, không tự đặt đơn thay khách.

**Lựa chọn 1 — Vì sao RAG thay vì fine-tune LLM:**

| Tiêu chí | Fine-tune LLM | RAG (chọn) |
|:---|:---|:---|
| Dữ liệu huấn luyện | Cần nhiều, gán nhãn | Không cần |
| Cập nhật tri thức | Huấn luyện lại | Sửa knowledge base |
| Trích nguồn (grounding) | Khó | Có — trả về citation |
| Nguy cơ bịa (hallucination) | Cao | Thấp (bị ràng buộc bởi context) |

**Lựa chọn 2 — Vì sao BM25 (lexical) thay vì dense embedding:**

| Tiêu chí | Dense embedding | BM25 / Okapi (chọn) |
|:---|:---|:---|
| Cần model + GPU + dữ liệu | Có | Không |
| Khả diễn giải (interpretability) | Thấp (vector ẩn) | Cao — điểm số từ TF, IDF, độ dài |
| Hợp corpus nhỏ, từ khóa rõ | Thừa | Đủ và mạnh |
| Tiếng Việt sau khi chuẩn hóa bỏ dấu | Cần model đa ngữ | Token ASCII khớp tốt |

→ BM25 là baseline mạnh, minh bạch, phù hợp ràng buộc. Dense/Hybrid được đề xuất ở phần **Hướng phát triển**, không nằm trong phạm vi đồ án.

**Khai phá dữ liệu trong bài toán (phần "Khai phá dữ liệu" của môn):**
- **Luật kết hợp (Association Rules):** cặp món hay đi cùng (món cay → đồ uống mát; món nặng → tráng miệng), lưu trong `combo-pairing.md` / `data-mining-insights.md`, dùng để gợi ý combo.
- **Gợi ý theo nội dung (Content-Based Recommendation):** đối chiếu khẩu vị khách ("mát", "cay", "ăn nhẹ") với **tag** và mô tả món.

**Phân tích rủi ro → thiết kế guardrails:** LLM có thể (a) tự nhận đã đặt đơn, (b) bịa giá/món ngoài thực đơn, (c) trả lời lệch chủ đề. Mỗi rủi ro được ánh xạ thành một cờ kiểm soát ở tầng input và một lớp kiểm duyệt schema ở tầng output (chi tiết Mục 5–6).

In [2]:
# ══════════════════════════════════════════════════
# 1.3 — Kiến trúc hệ thống
# ══════════════════════════════════════════════════

display(HTML("""
<div style='background:#08233E;border-radius:16px;padding:32px;color:#fff;font-family:monospace;font-size:13px;
            line-height:1.7;overflow-x:auto'>
<pre style='margin:0;color:#CFEAF5'>
  ┌──────────────────┐
  │  <span style='color:#08BCEF;font-weight:bold'>Customer Web</span>     │   React Chat UI, Quick Prompts,
  │  (React + Vite)  │   SuggestedCartActionCard
  └────────┬─────────┘
           │ HTTP POST /api/chat/sessions/{id}/messages
           ▼
  ┌──────────────────┐
  │  <span style='color:#08BCEF;font-weight:bold'>.NET Backend</span>     │   Session management, Menu injection,
  │  (ASP.NET API)   │   Authentication, Order creation
  └────────┬─────────┘
           │ HTTP POST /v1/chat
           ▼
  ┌──────────────────────────────────────────────────────┐
  │  <span style='color:#08BCEF;font-weight:bold'>Python AI Service</span> (FastAPI)                          │
  │                                                      │
  │   ┌─────────────┐    ┌─────────────────┐             │
  │   │ <span style='color:#FFD700'>Knowledge</span>   │───▶│ <span style='color:#FFD700'>BM25 Retriever</span> │             │
  │   │ <span style='color:#FFD700'>Base (.md)</span>  │    │ (Okapi BM25)    │             │
  │   └─────────────┘    └────────┬────────┘             │
  │                               │ top-K chunks         │
  │   ┌─────────────┐             ▼                      │
  │   │ <span style='color:#00FF88'>Guardrails</span>  │───▶ <span style='color:#FFD700'>Prompt Builder</span>                │
  │   │ (Input)     │    (System + RAG + Menu + History)  │
  │   └─────────────┘             │                      │
  │                               ▼                      │
  │                      ┌─────────────────┐             │
  │                      │ <span style='color:#FF6B6B'>9router Gateway</span> │             │
  │                      │ → Gemini 3.1 Pro│             │
  │                      └────────┬────────┘             │
  │                               │ JSON response        │
  │                               ▼                      │
  │   ┌─────────────┐    ┌─────────────────┐             │
  │   │ <span style='color:#00FF88'>Guardrails</span>  │◀───│ <span style='color:#FFD700'>Output Parser</span>  │             │
  │   │ (Output)    │    │ + Validation    │             │
  │   └─────────────┘    └─────────────────┘             │
  └──────────────────────────────────────────────────────┘
</pre>
</div>
"""))

### 1.4 Luồng xử lý chi tiết

```
Khách hỏi "Gợi ý món cho 2 người"
    │
    ├─ 1. Input Guardrails: detect intent (clean ✓)
    ├─ 2. BM25 Search: tìm top-5 chunks liên quan
    ├─ 3. Build Prompt: System Policy + RAG Context + Menu + History + User Message
    ├─ 4. Call LLM: Gemini 3.1 Pro → JSON response
    ├─ 5. Output Parser: extract JSON, validate menu IDs
    ├─ 6. Output Guardrails: force confirmation, block hallucination
    └─ 7. Response: text + suggested_cart_actions + guardrail_flags
```

<a id='sec2'></a>
<div style='border-bottom:3px solid #005BAC;padding-bottom:8px;margin:40px 0 20px'>
  <h2 style='color:#08233E;margin:0'>2. Knowledge Base — Kho tri thức nhà hàng</h2>
</div>

Knowledge base chứa thông tin nhà hàng dưới dạng **Markdown files**, được split theo heading `#` thành các **chunks** để retriever có thể tìm kiếm.

### Tại sao dùng Markdown?
- **Dễ cập nhật**: Nhân viên có thể sửa trực tiếp
- **Structured**: Heading `#` tự nhiên phân chia thành chunks
- **Lightweight**: Không cần database riêng

In [3]:
# ══════════════════════════════════════════════════
# 2.1 — Knowledge Base Loader (self-contained)
# ══════════════════════════════════════════════════

from dataclasses import dataclass

@dataclass(frozen=True)
class KnowledgeChunk:
    """Một đoạn tri thức nhỏ từ knowledge base."""
    source: str      # tên file gốc
    title: str       # tiêu đề heading
    content: str     # nội dung text
    tags: tuple      # tags trích từ filename

    @property
    def citation(self):
        return f"{self.source}::{self.title}"


# ── Dữ liệu Knowledge Base nhúng trực tiếp ──
# Mirror nguyên văn 7 file trong ai/knowledge-base/ của project.

KNOWLEDGE_BASE_RAW = {
    "menu.md": """
# Menu Và Quy Tắc Gợi Ý Món
AI chỉ được gợi ý món có trong menu backend gửi sang request hoặc món có trong knowledge base đã được duyệt. Không tự tạo tên món, giá, combo hoặc ưu đãi mới.

## Nhóm Món Chính
- Cơm gà xối mỡ: món chính phổ biến, hợp khách muốn ăn no nhanh.
- Cơm sườn nướng: món chính vị đậm, hợp bữa trưa hoặc bữa tối.
- Phở bò tái: món nước nóng, hợp khách muốn món nhẹ nhưng đủ no.
- Bún bò Huế: món cay, chỉ gợi ý nếu trạng thái còn hàng.

## Khai Vị Và Món Nhẹ
- Gỏi cuốn tôm thịt: món nhẹ, ít dầu, hợp ăn kèm.
- Chả giò hải sản: món chiên giòn, hợp đi cùng món nước hoặc đồ uống mát.

## Đồ Uống Và Tráng Miệng
- Trà đào cam sả: đồ uống mát, hợp món cay hoặc món nướng.
- Cà phê sữa đá: đồ uống cà phê, hợp khách muốn tỉnh táo.
- Chè khúc bạch: tráng miệng mát, hợp sau món chính.
- Bánh flan caramel: chỉ gợi ý nếu trạng thái còn hàng.

## Quy Tắc Giá
Giá món phải lấy từ backend hoặc dữ liệu menu đã duyệt. Nếu không có giá trong context, AI phải nói rằng hệ thống chưa có đủ thông tin giá.
""",

    "faq.md": """
# FAQ CMC Restaurant

## Nhà hàng phục vụ gì?
CMC Restaurant phục vụ món Việt phổ biến, đồ uống và tráng miệng. Khách có thể xem menu, thêm món vào giỏ và gửi đơn qua hệ thống QR.

## AI có đặt món thay khách không?
Không. AI chỉ tư vấn và đề xuất. Khách phải tự xác nhận trong giao diện trước khi đơn được gửi đi.

## Bếp có nhận đơn mang về không?
Có, nếu backend tạo đơn với loại mang về và đơn đã được xác nhận. AI không tự gửi đơn sang bếp.

## Có thể hỏi món phù hợp cho nhóm không?
Có. AI có thể gợi ý dựa trên số người, khẩu vị, món còn hàng và các cặp món thường đi cùng nhau.

## Giờ mở cửa của nhà hàng?
CMC Restaurant mở cửa từ 10:00 sáng đến 22:00 tối, phục vụ cả bữa trưa và bữa tối. Chủ nhật nghỉ.

## Nhà hàng có wifi không?
Có, nhà hàng cung cấp wifi miễn phí cho khách. Mật khẩu wifi được dán tại bàn hoặc hỏi nhân viên.

## Có chỗ đậu xe không?
Nhà hàng có bãi giữ xe máy miễn phí ngay trước cửa. Ô tô có thể đậu tại bãi xe công cộng cách 50m.

## Nhà hàng có nhận đặt bàn trước không?
Hiện tại nhà hàng phục vụ theo thứ tự đến trước. Chưa hỗ trợ đặt bàn trước qua hệ thống.

## Thanh toán bằng hình thức nào?
Khách có thể thanh toán tiền mặt tại quầy hoặc chuyển khoản qua VietQR. Hệ thống tạo mã QR thanh toán tự động.

## Có thể huỷ đơn sau khi gửi không?
Đơn đã gửi cho bếp không thể huỷ qua hệ thống. Khách cần gọi nhân viên trực tiếp nếu muốn thay đổi.
""",

    "ordering-policy.md": """
# Chính Sách Đặt Món
AI là trợ lý tư vấn, không phải nhân viên xác nhận đơn hàng.

## Quy Tắc An Toàn
- AI không tự tạo đơn hàng.
- AI không tự thêm món vào giỏ.
- AI không tự thanh toán.
- AI chỉ được đề xuất món và yêu cầu khách xác nhận thao tác.
- Backend .NET chịu trách nhiệm kiểm tra món tồn tại, giá, trạng thái còn hàng và quyền thao tác.

## Mang Về
Khách có thể đặt mang về nếu hệ thống hỗ trợ order type pickup. Đơn mang về vẫn phải đi qua backend và bếp như đơn bình thường sau khi khách xác nhận.

## Món Hết Hàng
Nếu món hết hàng, AI phải từ chối gợi ý món đó và đề xuất món thay thế đang còn hàng.
""",

    "allergy-dietary.md": """
# Thông Tin Dị Ứng Và Ăn Kiêng

## Dị Ứng Phổ Biến
- Hải sản: Chả giò hải sản có chứa tôm. Khách dị ứng tôm hoặc cua nên tránh món này.
- Đậu phộng: Một số món có nước chấm đậu phộng. AI nên hỏi lại nếu khách đề cập dị ứng.
- Gluten: Các món chiên có thể chứa bột mì. Khách cần ăn không gluten nên chọn phở hoặc bún.

## Ăn Chay
Hiện tại menu chưa có nhóm món chay riêng. Nếu khách hỏi món chay, AI nên nói rõ menu chưa có và đề xuất món nhẹ hoặc đồ uống.

## Ăn Kiêng, Ít Calo
- Món nhẹ ít calo: Gỏi cuốn tôm thịt (cuốn tươi, không chiên).
- Tránh món chiên nếu muốn giảm calo: Chả giò hải sản, Cơm gà xối mỡ (có gà chiên).
- Đồ uống không đường: Trà đào cam sả có thể yêu cầu ít đường.

## Lưu Ý Cho AI
AI không phải bác sĩ dinh dưỡng. Nếu khách hỏi sâu về y tế hoặc dinh dưỡng, nên khuyên khách tham khảo chuyên gia và chỉ cung cấp thông tin menu có sẵn.
""",

    "combo-pairing.md": """
# Gợi Ý Combo Và Cặp Món

## Combo Bữa Trưa 1 Người
- Cơm gà xối mỡ + Trà đào cam sả: tiết kiệm, phổ biến nhất.
- Phở bò tái + Cà phê sữa đá: hợp khách muốn ăn nhẹ nhưng cần tỉnh táo.
- Cơm sườn nướng + Chè khúc bạch: vị đậm kèm tráng miệng mát.

## Combo 2 Người Chia Sẻ
- 1 Cơm gà xối mỡ + 1 Cơm sườn nướng + 1 Gỏi cuốn tôm thịt + 2 Trà đào cam sả: đa dạng và cân đối.
- 1 Bún bò Huế + 1 Phở bò tái + 1 Chả giò hải sản: dành cho người thích khám phá.

## Combo Nhóm 4 Người
- 2 Cơm gà + 1 Cơm sườn + 1 Bún bò Huế + 1 Gỏi cuốn + 1 Chả giò + 4 Trà đào: gợi ý tiêu chuẩn cho nhóm bạn.

## Cặp Món Thường Đi Cùng (Association Rules)
Dữ liệu khai phá cho thấy:
- Món chính cay (Bún bò Huế) → thường gọi kèm đồ uống mát (Trà đào cam sả).
- Món chính nặng (Cơm sườn nướng) → thường gọi kèm tráng miệng (Chè khúc bạch).
- Khai vị (Gỏi cuốn) → thường gọi khi nhóm ≥2 người.
- Cà phê sữa đá → thường gọi đơn lẻ hoặc kèm phở.

## Quy Tắc Gợi Ý
AI chỉ gợi ý combo khi khách hỏi. Không tự push combo. Mỗi món trong combo phải có trong menu và đang còn hàng.
""",

    "data-mining-insights.md": """
# Insight Từ Học Máy Và Khai Phá Dữ Liệu
Knowledge base này liên kết với notebook trình bày trong ai/notebooks/ai_rag_presentation.ipynb.

## Association Rules
Các luật kết hợp có thể dùng để gợi ý món đi kèm, nhưng phải kiểm tra menu hiện tại trước khi trả lời.
Ví dụ:
- Món chính thường đi cùng đồ uống mát.
- Món cay nên gợi ý kèm trà hoặc món tráng miệng mát.
- Món khai vị nhẹ phù hợp khi khách hỏi gợi ý cho 2 người.

## Content-Based Recommendation
Khi khách nói khẩu vị như mát, cay, ăn nhẹ, ăn no, AI nên đối chiếu tag món và mô tả món trong menu.

## Giới Hạn
Insight khai phá dữ liệu là tín hiệu hỗ trợ, không thay thế dữ liệu menu thật. Nếu dữ liệu mâu thuẫn, ưu tiên menu backend và trạng thái availability.
""",

    "brand-voice.md": """
# Phong Cách Trả Lời

## Giọng Văn
- Lịch sự, ấm áp, rõ ràng.
- Tiếng Việt có dấu.
- Không quá dài.
- Ưu tiên câu trả lời có thể thao tác ngay.

## Mẫu Trả Lời Tốt
Nếu bạn ăn trưa 2 người, mình gợi ý Cơm sườn nướng, Gỏi cuốn tôm thịt và Trà đào cam sả. Bạn kiểm tra lại giỏ hàng rồi xác nhận đặt món trên giao diện nhé.

## Mẫu Trả Lời Không Được Dùng
Mình đã đặt đơn cho bạn.
Lý do: AI không được tự tạo đơn hàng.
"""
}


def split_markdown(source_name, text):
    """Split markdown text bằng headings thành list of KnowledgeChunk."""
    lines = text.strip().splitlines()
    chunks = []
    tags = tuple(p for p in source_name.replace(".md","").replace("_","-").split("-") if p)
    current_title = source_name.replace(".md","").replace("-"," ").title()
    current_lines = []

    def flush():
        content = "\n".join(l.strip() for l in current_lines).strip()
        if content:
            chunks.append(KnowledgeChunk(source=source_name, title=current_title, content=content, tags=tags))

    for line in lines:
        if line.startswith("#"):
            flush()
            current_title = line.lstrip("#").strip() or current_title
            current_lines = []
        else:
            current_lines.append(line)
    flush()
    return chunks


# Load tất cả chunks
chunks = []
for name, text in sorted(KNOWLEDGE_BASE_RAW.items()):
    chunks.extend(split_markdown(name, text))

print(f"Loaded {len(chunks)} chunks from {len(KNOWLEDGE_BASE_RAW)} files")

Loaded 35 chunks from 7 files


In [4]:
# ══════════════════════════════════════════════════
# 2.2 — Hiển thị Knowledge Base
# ══════════════════════════════════════════════════
from collections import Counter

source_counts = Counter(c.source for c in chunks)
desc_map = {
    "menu.md": ("Thông tin món ăn, nhóm món, quy tắc giá", "#005BAC"),
    "faq.md": ("Câu hỏi thường gặp: giờ mở cửa, wifi, thanh toán", "#08BCEF"),
    "ordering-policy.md": ("Chính sách đặt món, quy tắc an toàn AI", "#FF6B6B"),
    "brand-voice.md": ("Giọng văn trả lời, mẫu câu tốt/cấm", "#FFB347"),
    "allergy-dietary.md": ("Dị ứng, ăn kiêng, ăn chay", "#00A86B"),
    "combo-pairing.md": ("Combo bữa ăn, cặp món (association rules)", "#9B59B6"),
    "data-mining-insights.md": ("Insight từ khai phá dữ liệu", "#E67E22"),
}

rows = ""
for source, count in sorted(source_counts.items()):
    desc, color = desc_map.get(source, ("", "#333"))
    bar = "<span style='color:" + color + "'>" + "█" * (count * 4) + "</span>"
    rows += f"""<tr>
      <td style='padding:10px 14px;font-weight:700'>{source}</td>
      <td style='text-align:center;padding:10px'><span style='background:{color};color:#fff;padding:3px 10px;
          border-radius:99px;font-size:13px;font-weight:700'>{count}</span></td>
      <td style='padding:10px 14px'>{desc}</td>
      <td style='padding:10px 14px'>{bar}</td>
    </tr>"""

display(HTML(f"""
<table style='border-collapse:collapse;width:100%;font-size:14px'>
<tr style='background:#08233E;color:#fff'>
  <th style='padding:10px 14px;text-align:left'>Source File</th>
  <th style='padding:10px'>Chunks</th>
  <th style='padding:10px 14px;text-align:left'>Mô tả</th>
  <th style='padding:10px 14px;text-align:left'>Distribution</th>
</tr>
{rows}
<tr style='background:#F2FAFD;font-weight:700'>
  <td style='padding:10px 14px'>TOTAL</td>
  <td style='text-align:center;padding:10px'>{len(chunks)}</td>
  <td colspan='2' style='padding:10px 14px'>{len(source_counts)} source files</td>
</tr>
</table>
"""))

In [5]:
# ══════════════════════════════════════════════════
# 2.3 — Chi tiết từng chunk (scrollable)
# ══════════════════════════════════════════════════
rows = ""
for i, c in enumerate(chunks):
    preview = c.content[:100].replace('\n',' ') + ('...' if len(c.content)>100 else '')
    bg = '#fff' if i%2==0 else '#F8FCFE'
    rows += f"""<tr style='background:{bg}'>
      <td style='padding:8px;text-align:center;color:#667A91'>{i+1}</td>
      <td style='padding:8px;font-weight:600;color:#005BAC;white-space:nowrap'>{c.source}</td>
      <td style='padding:8px;font-weight:600'>{c.title}</td>
      <td style='padding:8px;color:#333;font-size:12px'>{preview}</td>
    </tr>"""

display(HTML(f"""
<div style='max-height:350px;overflow-y:auto;border:1px solid #CFEAF5;border-radius:12px'>
<table style='border-collapse:collapse;width:100%;font-size:13px'>
<tr style='background:#08233E;color:#fff;position:sticky;top:0'>
  <th style='padding:10px 8px'>#</th><th style='padding:10px 8px;text-align:left'>Source</th>
  <th style='padding:10px 8px;text-align:left'>Title</th><th style='padding:10px 8px;text-align:left'>Content</th>
</tr>{rows}</table></div>
"""))

#,Source,Title,Content
1,allergy-dietary.md,Dị Ứng Phổ Biến,- Hải sản: Chả giò hải sản có chứa tôm. Khách dị ứng tôm hoặc cua nên tránh món này. - Đậu phộng: Mộ...
2,allergy-dietary.md,Ăn Chay,"Hiện tại menu chưa có nhóm món chay riêng. Nếu khách hỏi món chay, AI nên nói rõ menu chưa có và đề ..."
3,allergy-dietary.md,"Ăn Kiêng, Ít Calo","- Món nhẹ ít calo: Gỏi cuốn tôm thịt (cuốn tươi, không chiên). - Tránh món chiên nếu muốn giảm calo:..."
4,allergy-dietary.md,Lưu Ý Cho AI,"AI không phải bác sĩ dinh dưỡng. Nếu khách hỏi sâu về y tế hoặc dinh dưỡng, nên khuyên khách tham kh..."
5,brand-voice.md,Giọng Văn,"- Lịch sự, ấm áp, rõ ràng. - Tiếng Việt có dấu. - Không quá dài. - Ưu tiên câu trả lời có thể thao t..."
6,brand-voice.md,Mẫu Trả Lời Tốt,"Nếu bạn ăn trưa 2 người, mình gợi ý Cơm sườn nướng, Gỏi cuốn tôm thịt và Trà đào cam sả. Bạn kiểm tr..."
7,brand-voice.md,Mẫu Trả Lời Không Được Dùng,Mình đã đặt đơn cho bạn. Lý do: AI không được tự tạo đơn hàng.
8,combo-pairing.md,Combo Bữa Trưa 1 Người,"- Cơm gà xối mỡ + Trà đào cam sả: tiết kiệm, phổ biến nhất. - Phở bò tái + Cà phê sữa đá: hợp khách ..."
9,combo-pairing.md,Combo 2 Người Chia Sẻ,- 1 Cơm gà xối mỡ + 1 Cơm sườn nướng + 1 Gỏi cuốn tôm thịt + 2 Trà đào cam sả: đa dạng và cân đối. -...
10,combo-pairing.md,Combo Nhóm 4 Người,- 2 Cơm gà + 1 Cơm sườn + 1 Bún bò Huế + 1 Gỏi cuốn + 1 Chả giò + 4 Trà đào: gợi ý tiêu chuẩn cho nh...


In [6]:
# ══════════════════════════════════════════════════
# 2.x — Phân tích Corpus (dataset của hệ thống RAG)
# Số liệu tính trực tiếp từ `chunks` đã load ở 2.1
# ══════════════════════════════════════════════════

from collections import Counter

# Ánh xạ mỗi file tri thức ↔ kỹ thuật / vai trò (góc nhìn IR & Khai phá dữ liệu)
ROLE = {
    "menu.md":                 "Dữ liệu miền + quy tắc giá (grounding)",
    "faq.md":                  "Hỏi-đáp vận hành",
    "ordering-policy.md":      "Ràng buộc an toàn đặt món",
    "allergy-dietary.md":      "Tri thức dị ứng / ăn kiêng",
    "combo-pairing.md":        "Luật kết hợp (Association Rules)",
    "data-mining-insights.md": "Luật kết hợp + Gợi ý theo nội dung",
    "brand-voice.md":          "Phong cách sinh câu trả lời",
}

per_file = Counter(c.source for c in chunks)
words = {c.source: 0 for c in chunks}
for c in chunks:
    words[c.source] += len(c.content.split())

rows = ""
for name in sorted(KNOWLEDGE_BASE_RAW):
    n = per_file.get(name, 0)
    avg = round(words.get(name, 0) / n) if n else 0
    rows += (f"<tr><td style='padding:6px 12px;font-family:monospace'>{name}</td>"
             f"<td style='padding:6px 12px;text-align:center'>{n}</td>"
             f"<td style='padding:6px 12px;text-align:center'>{avg}</td>"
             f"<td style='padding:6px 12px;color:#667A91'>{ROLE.get(name,'-')}</td></tr>")

total_words = sum(words.values())
display(HTML(f"""
<table style='border-collapse:collapse;width:100%;font-size:13px;border:1px solid #CFEAF5'>
<tr style='background:#08233E;color:#fff'>
  <th style='padding:8px 12px;text-align:left'>File</th><th style='padding:8px'>Chunks</th>
  <th style='padding:8px'>~Từ/chunk</th><th style='padding:8px 12px;text-align:left'>Vai trò / Kỹ thuật</th></tr>
{rows}
<tr style='background:#EAF7FC;font-weight:700'>
  <td style='padding:8px 12px'>TỔNG: {len(KNOWLEDGE_BASE_RAW)} docs</td>
  <td style='padding:8px;text-align:center'>{len(chunks)}</td>
  <td style='padding:8px;text-align:center'>{round(total_words/len(chunks))}</td>
  <td style='padding:8px 12px'>{total_words} từ</td></tr>
</table>
<p style='font-size:12px;color:#667A91;margin:8px 0 0'>Mỗi heading markdown → một chunk; tags trích từ tên file để boost xếp hạng (Mục 3).</p>
"""))

File,Chunks,~Từ/chunk,Vai trò / Kỹ thuật
allergy-dietary.md,4,42,Tri thức dị ứng / ăn kiêng
brand-voice.md,3,26,Phong cách sinh câu trả lời
combo-pairing.md,5,44,Luật kết hợp (Association Rules)
data-mining-insights.md,4,32,Luật kết hợp + Gợi ý theo nội dung
faq.md,10,23,Hỏi-đáp vận hành
menu.md,5,40,Dữ liệu miền + quy tắc giá (grounding)
ordering-policy.md,4,31,Ràng buộc an toàn đặt món
TỔNG: 7 docs,35,33,1146 từ


<a id='sec3'></a>
<div style='border-bottom:3px solid #005BAC;padding-bottom:8px;margin:40px 0 20px'>
  <h2 style='color:#08233E;margin:0'>3. BM25 Retriever — Truy xuất thông tin</h2>
</div>

### 3.1 Okapi BM25 — Thuật toán

**BM25** là thuật toán ranking tiêu chuẩn trong Information Retrieval (Robertson et al., 1994):

$$\text{Score}(q, D) = \sum_{t \in q} \underbrace{\log\frac{N - df(t) + 0.5}{df(t) + 0.5}}_{\text{IDF}(t)} \cdot \underbrace{\frac{tf(t,D) \cdot (k_1 + 1)}{tf(t,D) + k_1 \cdot \left(1 - b + b \cdot \frac{|D|}{\text{avgdl}}\right)}}_{\text{TF saturation + length norm}}$$

| Tham số | Giá trị | Ý nghĩa |
|:--------|:--------|:--------|
| **k₁** | 1.5 | Kiểm soát term frequency saturation |
| **b** | 0.75 | Kiểm soát document length normalization |
| **Title boost** | +1.5 | Ưu tiên chunk có title match |
| **Tag boost** | +1.0 | Ưu tiên chunk có tag match |

In [7]:
# ══════════════════════════════════════════════════
# 3.2 — BM25 Retriever Implementation (self-contained)
# ══════════════════════════════════════════════════

import math, re, unicodedata

TOKEN_RE = re.compile(r"[a-z0-9]+", re.IGNORECASE)
BM25_K1, BM25_B = 1.5, 0.75
TITLE_BOOST, TAG_BOOST = 1.5, 1.0


def tokenize(text):
    """Chuẩn hóa tiếng Việt → ASCII lowercase → tách tokens."""
    norm = unicodedata.normalize("NFKD", text.lower()).replace("đ", "d")
    ascii_text = "".join(ch for ch in norm if not unicodedata.combining(ch))
    return TOKEN_RE.findall(ascii_text)


class BM25Retriever:
    """Okapi BM25 retriever with title/tag boosting."""

    def __init__(self, chunks):
        self.chunks = chunks
        self.doc_tokens = [tokenize(c.title + " " + c.content + " " + " ".join(c.tags)) for c in chunks]
        self.doc_sets = [set(t) for t in self.doc_tokens]
        self.N = len(chunks)
        self.avgdl = sum(len(t) for t in self.doc_tokens) / max(self.N, 1)
        # document frequency
        self.df = {}
        for s in self.doc_sets:
            for tok in s:
                self.df[tok] = self.df.get(tok, 0) + 1

    def search(self, query, top_k=5):
        q_tokens = set(tokenize(query))
        if not q_tokens:
            return []
        scored = []
        for idx, chunk in enumerate(self.chunks):
            overlap = q_tokens & self.doc_sets[idx]
            if not overlap:
                continue
            doc_len = len(self.doc_tokens[idx])
            tf_map = {}
            for tok in self.doc_tokens[idx]:
                tf_map[tok] = tf_map.get(tok, 0) + 1
            score = 0.0
            for tok in overlap:
                tf = tf_map.get(tok, 0)
                df = self.df.get(tok, 0)
                idf = math.log((self.N - df + 0.5) / (df + 0.5) + 1.0)
                score += idf * (tf * (BM25_K1 + 1)) / (tf + BM25_K1 * (1 - BM25_B + BM25_B * doc_len / self.avgdl))
            # Boosts
            score += TITLE_BOOST * len(q_tokens & set(tokenize(chunk.title)))
            score += TAG_BOOST * len(q_tokens & set(chunk.tags))
            scored.append((chunk, round(score, 4)))
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:top_k]


retriever = BM25Retriever(chunks)

display(HTML(
    "<div style='display:flex;flex-wrap:wrap;margin:8px 0'>" +
    info_card("Corpus", f"{retriever.N}", "chunks") +
    info_card("Vocabulary", f"{len(retriever.df):,}", "unique tokens") +
    info_card("Avg Length", f"{retriever.avgdl:.0f}", "tokens/chunk") +
    info_card("BM25", "k1=1.5", "b=0.75", "#08BCEF") +
    "</div>"
))

In [8]:
# ══════════════════════════════════════════════════
# 3.3 — Demo: BM25 Search
# ══════════════════════════════════════════════════

demo_queries = [
    ("Gợi ý món cho 2 người ăn trưa",    "Combo"),
    ("Tôi bị dị ứng hải sản",             "Dị ứng"),
    ("Nhà hàng mở cửa mấy giờ?",         "FAQ"),
    ("Món cay nên uống gì kèm?",          "Association rules"),
    ("Thanh toán bằng cách nào?",          "Thanh toán"),
    ("Có món chay không?",                 "Dietary"),
]

for query, category in demo_queries:
    results = retriever.search(query, top_k=3)
    result_html = ""
    for rank, (chunk, score) in enumerate(results, 1):
        bar_w = min(score * 8, 200)
        result_html += f"""
        <div style='display:grid;grid-template-columns:30px 1fr 90px;gap:8px;align-items:center;
                    padding:6px 0;border-bottom:1px solid #EAF7FC'>
          <span style='color:#005BAC;font-weight:900;font-size:16px'>#{rank}</span>
          <div><b style='color:#08233E'>{chunk.title}</b>
            <span style='color:#667A91;font-size:12px;margin-left:6px'>({chunk.source})</span></div>
          <div style='text-align:right'>
            <div style='background:linear-gradient(90deg,#005BAC,#08BCEF);height:8px;border-radius:4px;
                        width:{bar_w}px;margin-left:auto'></div>
            <span style='font-size:11px;color:#667A91'>{score:.2f}</span>
          </div>
        </div>"""
    display(HTML(f"""
    <div style='border:1px solid #CFEAF5;border-radius:14px;padding:16px 20px;margin-bottom:10px;background:#fff'>
      <div style='display:flex;justify-content:space-between;align-items:center;margin-bottom:8px'>
        <span><b>Query:</b> "{query}"</span>
        <span style='background:#EAF7FC;color:#005BAC;padding:3px 12px;border-radius:99px;
               font-size:11px;font-weight:700'>{category}</span>
      </div>{result_html}
    </div>"""))

<a id='sec4'></a>
<div style='border-bottom:3px solid #005BAC;padding-bottom:8px;margin:40px 0 20px'>
  <h2 style='color:#08233E;margin:0'>4. Prompt Engineering — Xây dựng ngữ cảnh cho LLM</h2>
</div>

| # | Role | Nội dung | Mục đích |
|:--|:-----|:---------|:---------|
| 1 | `system` | System Policy | Quy tắc an toàn, JSON schema |
| 2 | `system` | RAG Context | Top-K chunks từ BM25 |
| 3 | `system` | Menu hiện có | Realtime menu từ database |
| 4 | varies | Chat History | 8 tin nhắn gần nhất |
| 5 | `user` | User Message | Câu hỏi hiện tại |

In [9]:
# ══════════════════════════════════════════════════
# 4.1 — System Policy & Prompt Builder (self-contained)
# ══════════════════════════════════════════════════

SYSTEM_POLICY = """Bạn là trợ lý AI của CMC Restaurant.
Chỉ trả lời dựa trên menu, FAQ, chính sách nhà hàng và RAG context được cung cấp.
Không bịa món, không bịa giá, không tự tạo đơn, không tự thêm món vào giỏ và không tự thanh toán.
Bạn chỉ được đề xuất món để khách xác nhận thủ công trong giao diện.
Nếu thiếu dữ liệu, hãy nói rõ hệ thống chưa có đủ thông tin.
Luôn trả về JSON hợp lệ, không markdown, không giải thích ngoài JSON.
Schema bắt buộc:
{
  "content": "Câu trả lời ngắn gọn bằng tiếng Việt có dấu.",
  "suggested_cart_actions": [
    {
      "menu_item_id": "id món có thật trong menu",
      "name": "tên món",
      "price_vnd": 65000,
      "quantity": 1,
      "reason": "lý do gợi ý",
      "requires_customer_confirmation": true
    }
  ],
  "guardrail_flags": ["CUSTOMER_CONFIRMATION_REQUIRED"]
}
Nếu không có món phù hợp, suggested_cart_actions phải là []."""


def build_prompt(user_message, context_results, menu_items, history=None):
    """Xây dựng prompt messages cho LLM."""
    context_text = "\n\n".join(
        f"[{i}] {chunk.citation}\n{chunk.content}"
        for i, (chunk, score) in enumerate(context_results, 1)
    ) or "Không có context phù hợp."

    menu_text = "\n".join(
        f"- {m['id']}: {m['name']}, giá {m.get('price_vnd','?')} VND, "
        f"{'còn' if m.get('is_available',True) else 'hết'}"
        for m in menu_items[:20]
    ) or "Menu chưa được cung cấp."

    msgs = [
        {"role": "system", "content": SYSTEM_POLICY},
        {"role": "system", "content": f"RAG context:\n{context_text}"},
        {"role": "system", "content": f"Menu hiện có:\n{menu_text}"},
    ]
    for h in (history or [])[-8:]:
        if h.get("content"):
            msgs.append({"role": h.get("role","user"), "content": h["content"]})
    msgs.append({"role": "user", "content": user_message})
    return msgs


display(HTML(f"""
<div style='background:#FFF8F0;border:1px solid #FFD700;border-radius:14px;padding:20px'>
  <h4 style='color:#E67E22;margin:0 0 12px'>System Policy — Luôn được inject đầu tiên</h4>
  <pre style='background:#08233E;color:#CFEAF5;padding:16px;border-radius:10px;font-size:12px;
             white-space:pre-wrap;line-height:1.6;margin:0'>{SYSTEM_POLICY}</pre>
</div>
"""))

In [10]:
# ══════════════════════════════════════════════════
# 4.2 — Demo xây dựng prompt hoàn chỉnh
# ══════════════════════════════════════════════════

MOCK_MENU = [
    {"id": "m_001", "name": "Cơm gà xối mỡ", "price_vnd": 45000, "is_available": True},
    {"id": "m_002", "name": "Phở bò tái", "price_vnd": 65000, "is_available": True},
    {"id": "m_003", "name": "Trà đào cam sả", "price_vnd": 30000, "is_available": True},
    {"id": "m_004", "name": "Cơm sườn nướng", "price_vnd": 55000, "is_available": True},
    {"id": "m_005", "name": "Gỏi cuốn tôm thịt", "price_vnd": 35000, "is_available": True},
    {"id": "m_006", "name": "Chả giò hải sản", "price_vnd": 40000, "is_available": True},
    {"id": "m_007", "name": "Bún bò Huế", "price_vnd": 60000, "is_available": False},
]

user_msg = "Gợi ý món cho 2 người ăn trưa"
context = retriever.search(user_msg, top_k=3)
messages = build_prompt(user_msg, context, MOCK_MENU)

colors = {"system": "#005BAC", "user": "#00A86B", "assistant": "#9B59B6"}
parts = ""
for i, msg in enumerate(messages):
    color = colors.get(msg['role'], '#333')
    content = msg['content'][:280] + ('...' if len(msg['content'])>280 else '')
    parts += f"""
    <div style='border-left:4px solid {color};padding:10px 16px;margin-bottom:8px;background:#fff;
                border-radius:0 10px 10px 0'>
      <div style='font-size:11px;font-weight:700;color:{color};text-transform:uppercase;
                  letter-spacing:0.06em;margin-bottom:4px'>Message {i+1} — {msg['role']}</div>
      <pre style='margin:0;font-size:12px;white-space:pre-wrap;color:#333;line-height:1.5'>{content}</pre>
    </div>"""

display(HTML(f"""
<div style='background:#F2FAFD;border-radius:14px;padding:20px'>
  <h4 style='margin:0 0 12px;color:#08233E'>Prompt cho: "{user_msg}" ({len(messages)} messages)</h4>
  {parts}
</div>"""))

<a id='sec5'></a>
<div style='border-bottom:3px solid #005BAC;padding-bottom:8px;margin:40px 0 20px'>
  <h2 style='color:#08233E;margin:0'>5. Guardrails — Hệ thống an toàn AI</h2>
</div>

### Bảo vệ 2 chiều

| Lớp | Thời điểm | Phương pháp |
|:----|:----------|:------------|
| **Input Guardrails** | Trước khi gọi LLM | Regex pattern matching trên text đã chuẩn hóa |
| **Output Guardrails** | Sau khi LLM trả lời | Validate menu IDs, force confirmation, clamp quantity |

In [11]:
# ══════════════════════════════════════════════════
# 5.1 — Guardrails Implementation (self-contained)
# Mirror nguyên văn ai/app/rag/guardrails.py
# ══════════════════════════════════════════════════

# 7 pattern ý định tạo đơn qua chat → buộc khách xác nhận
ORDER_CREATION_PATTERNS = [
    r"\bdat\s+luon\b", r"\bdat\s+mon\b", r"\bthem\s+vao\s+gio\b",
    r"\bthanh\s+toan\b", r"\bchot\s+don\b", r"\bgui\s+don\b", r"\bmua\s+luon\b",
]
# 11 pattern lệch chủ đề nhà hàng → OUT_OF_SCOPE
OFF_TOPIC_PATTERNS = [
    r"\bthoi\s+tiet\b", r"\bbong\s+da\b", r"\bchinh\s+tri\b", r"\btin\s+tuc\b",
    r"\bchung\s+khoan\b", r"\bcrypto\b", r"\bbitcoin\b", r"\blam\s+bai\b",
    r"\bgiai\s+toan\b", r"\bviet\s+code\b", r"\blap\s+trinh\b",
]
# 5 pattern thô tục → PROFANITY_DETECTED
PROFANITY_PATTERNS = [r"\bdm\b", r"\bvcl\b", r"\bngu\b", r"\bdien\b.*\bchung\b", r"\bmat\s+day\b"]


def normalize_vi(text):
    """Chuẩn hóa tiếng Việt: bỏ dấu, lowercase (mirror _normalize)."""
    norm = unicodedata.normalize("NFKD", text.lower().replace("đ", "d"))
    return "".join(ch for ch in norm if not unicodedata.combining(ch))


def detect_guardrail_flags(message):
    """Phát hiện intent nguy hiểm trong tin nhắn (5 cờ input guardrail)."""
    n = normalize_vi(message)
    flags = []
    if any(re.search(p, n) for p in ORDER_CREATION_PATTERNS):
        flags.append("CUSTOMER_CONFIRMATION_REQUIRED")
    if "gia" in n and ("tu tao" in n or "bia" in n or "re hon" in n):
        flags.append("PRICE_FABRICATION_BLOCKED")
    if "mon moi" in n or "ngoai thuc don" in n or "tu nghi" in n:
        flags.append("MENU_FABRICATION_BLOCKED")
    if any(re.search(p, n) for p in OFF_TOPIC_PATTERNS):
        flags.append("OUT_OF_SCOPE")
    if any(re.search(p, n) for p in PROFANITY_PATTERNS):
        flags.append("PROFANITY_DETECTED")
    return flags


# Demo
test_cases = [
    ("Gợi ý món cho 2 người",                "Normal",    "#00A86B"),
    ("Bạn đặt luôn cơm sườn cho tôi nhé",   "Order",     "#FF6B6B"),
    ("Thanh toán giúp tôi luôn đi",          "Payment",   "#FF6B6B"),
    ("Bịa giúp tôi giá rẻ hơn",             "Price hack","#FF6B6B"),
    ("Tạo món mới ngoài thực đơn",           "Menu hack", "#FF6B6B"),
    ("Hôm nay bóng đá có trận nào?",         "Off-topic", "#E67E22"),
    ("Viết code Python giúp tôi",            "Off-topic", "#E67E22"),
    ("Chốt đơn cho tôi luôn đi",             "Order",     "#FF6B6B"),
]

rows = ""
for msg, cat, color in test_cases:
    flags = detect_guardrail_flags(msg)
    badges = " ".join(f"<span style='background:#FFF0F0;color:#FF6B6B;padding:2px 8px;border-radius:6px;font-size:11px;font-weight:700'>{f}</span>" for f in flags) if flags else "<span style='background:#F0FFF4;color:#00A86B;padding:2px 8px;border-radius:6px;font-size:11px;font-weight:700'>CLEAN</span>"
    rows += f"<tr><td style='padding:10px 14px'>{msg}</td><td style='padding:10px;text-align:center'><span style='background:{color}22;color:{color};padding:2px 10px;border-radius:6px;font-size:11px;font-weight:700'>{cat}</span></td><td style='padding:10px 14px'>{badges}</td></tr>"

display(HTML(f"""
<table style='border-collapse:collapse;width:100%;font-size:14px'>
<tr style='background:#08233E;color:#fff'>
  <th style='padding:10px 14px;text-align:left'>Tin nhắn</th>
  <th style='padding:10px'>Loại</th>
  <th style='padding:10px 14px;text-align:left'>Guardrail Flags</th>
</tr>{rows}</table>
"""))

Tin nhắn,Loại,Guardrail Flags
Gợi ý món cho 2 người,Normal,CLEAN
Bạn đặt luôn cơm sườn cho tôi nhé,Order,CUSTOMER_CONFIRMATION_REQUIRED
Thanh toán giúp tôi luôn đi,Payment,CUSTOMER_CONFIRMATION_REQUIRED
Bịa giúp tôi giá rẻ hơn,Price hack,PRICE_FABRICATION_BLOCKED
Tạo món mới ngoài thực đơn,Menu hack,MENU_FABRICATION_BLOCKED
Hôm nay bóng đá có trận nào?,Off-topic,OUT_OF_SCOPE
Viết code Python giúp tôi,Off-topic,OUT_OF_SCOPE
Chốt đơn cho tôi luôn đi,Order,CUSTOMER_CONFIRMATION_REQUIRED


<a id='sec6'></a>
<div style='border-bottom:3px solid #005BAC;padding-bottom:8px;margin:40px 0 20px'>
  <h2 style='color:#08233E;margin:0'>6. Output Parser — Xử lý & Kiểm duyệt đầu ra</h2>
</div>

In [12]:
# ══════════════════════════════════════════════════
# 6.1 — Output Parser Implementation (self-contained)
# Mirror nguyên văn ai/app/rag/output_parser.py
# ══════════════════════════════════════════════════
import json

def parse_llm_response(raw, menu_items):
    """Parse + validate JSON response từ LLM."""
    # 1. Extract JSON
    text = (raw or "").strip()
    payload = None
    for candidate in [text, text[text.find('{'):text.rfind('}')+1] if '{' in text else ""]:
        try:
            p = json.loads(candidate)
            if isinstance(p, dict): payload = p; break
        except: pass
    if not payload or not str(payload.get("content","")).strip():
        return None

    # 2. Build available menu index
    avail = {str(m.get('id','')).strip(): m for m in menu_items
             if str(m.get('id','')).strip() and m.get('is_available', True)}

    # 3. Validate suggested_cart_actions (mirror _parse_suggested_actions)
    sca = payload.get("suggested_cart_actions")
    actions, flags = [], []
    # merge cờ guardrail mà LLM tự khai (chuẩn hóa upper)
    flags += [str(f).strip().upper() for f in (payload.get("guardrail_flags") or []) if str(f).strip()]

    if sca is not None and not isinstance(sca, list):
        # LLM trả sai schema (vd: dict/string thay vì list) → chặn cứng
        flags.append("AI_OUTPUT_SCHEMA_INVALID")
    else:
        rejected = 0
        for item in (sca or []):
            if not isinstance(item, dict):
                rejected += 1; continue
            mid = str(item.get('menu_item_id','')).strip()
            menu_match = avail.get(mid)
            if not menu_match:
                rejected += 1; continue          # món bịa hoặc hết hàng
            qty = max(1, min(int(item.get('quantity',1) or 1), 20))
            actions.append({
                "menu_item_id": mid,
                "name": item.get('name') or menu_match.get('name',''),
                "price_vnd": item.get('price_vnd') or menu_match.get('price_vnd'),
                "quantity": qty,
                "reason": item.get('reason'),
                "requires_customer_confirmation": True,  # ALWAYS forced True
            })
        if actions: flags.append("CUSTOMER_CONFIRMATION_REQUIRED")
        if rejected: flags.append("MENU_FABRICATION_BLOCKED")

    # dedupe giữ thứ tự
    seen, deduped = set(), []
    for f in flags:
        if f and f not in seen: seen.add(f); deduped.append(f)
    return {"content": str(payload['content']).strip(), "actions": actions, "flags": deduped}


# ── Demo ──
mock_llm = json.dumps({
    "content": "Cho 2 người ăn trưa, mình gợi ý Cơm gà xối mỡ và Trà đào cam sả.",
    "suggested_cart_actions": [
        {"menu_item_id":"m_001","name":"Cơm gà xối mỡ","price_vnd":45000,"quantity":2,"reason":"Món chính phổ biến","requires_customer_confirmation":False},
        {"menu_item_id":"m_003","name":"Trà đào cam sả","price_vnd":30000,"quantity":2,"reason":"Đồ uống mát","requires_customer_confirmation":False},
        {"menu_item_id":"m_999","name":"Món bịa","price_vnd":99000,"quantity":1,"reason":"LLM hallucinated"},
        {"menu_item_id":"m_007","name":"Bún bò Huế","price_vnd":60000,"quantity":1,"reason":"Hết hàng"},
    ],
    "guardrail_flags": []
}, ensure_ascii=False)

parsed = parse_llm_response(mock_llm, MOCK_MENU)
raw_actions = json.loads(mock_llm)['suggested_cart_actions']

# Edge case: LLM trả suggested_cart_actions sai schema → AI_OUTPUT_SCHEMA_INVALID
bad_schema = json.dumps({"content": "ok", "suggested_cart_actions": {"oops": "not a list"}}, ensure_ascii=False)
schema_flags = parse_llm_response(bad_schema, MOCK_MENU)['flags']
print("Schema-invalid demo → flags:", schema_flags)

# Visualize
in_rows = ""
for item in raw_actions:
    mid = item.get('menu_item_id','')
    mm = next((m for m in MOCK_MENU if m['id']==mid), None)
    if mm is None: st = "<span style='color:#FF6B6B;font-weight:700'>BLOCKED (hallucinated)</span>"
    elif not mm.get('is_available',True): st = "<span style='color:#E67E22;font-weight:700'>BLOCKED (hết hàng)</span>"
    else: st = "<span style='color:#00A86B;font-weight:700'>PASSED</span>"
    conf = item.get('requires_customer_confirmation',True)
    cf = f"<span style='color:#FF6B6B'>{conf} -> <b>True</b></span>" if not conf else "True"
    in_rows += f"<tr><td style='padding:8px;font-family:monospace;font-weight:700'>{mid}</td><td style='padding:8px'>{item.get('name','')}</td><td style='padding:8px;text-align:center'>{item.get('quantity',1)}</td><td style='padding:8px'>{cf}</td><td style='padding:8px'>{st}</td></tr>"

out_html = "".join(f"<div style='padding:6px 0;border-bottom:1px solid #D0F0D0'><b>{a['name']}</b> x{a['quantity']} @ {a['price_vnd']}d<br><span style='font-size:12px;color:#667A91'>confirmation: {a['requires_customer_confirmation']}</span></div>" for a in parsed['actions'])
flag_html = "".join(f"<span style='background:#FFF0F0;color:#FF6B6B;padding:2px 8px;border-radius:6px;font-size:11px;font-weight:700;margin-right:4px'>{f}</span>" for f in parsed['flags'])

display(HTML(f"""
<div style='display:grid;grid-template-columns:1fr auto 1fr;gap:16px;align-items:start'>
  <div><h4 style='color:#FF6B6B;margin:0 0 8px'>LLM Raw Output ({len(raw_actions)} items)</h4>
    <table style='border-collapse:collapse;width:100%;font-size:12px;border:1px solid #CFEAF5'>
    <tr style='background:#08233E;color:#fff'><th style='padding:6px'>ID</th><th style='padding:6px'>Name</th><th style='padding:6px'>Qty</th><th style='padding:6px'>Confirm</th><th style='padding:6px'>Status</th></tr>
    {in_rows}</table></div>
  <div style='display:flex;align-items:center;font-size:32px;color:#005BAC;padding-top:50px'>-></div>
  <div><h4 style='color:#00A86B;margin:0 0 8px'>After Validation ({len(parsed['actions'])} items)</h4>
    <div style='border:2px solid #00A86B;border-radius:10px;padding:12px;background:#F0FFF4'>{out_html}</div>
    <div style='margin-top:8px'><span style='font-size:12px;color:#667A91'>Flags: </span>{flag_html}</div></div>
</div>"""))

Schema-invalid demo → flags: ['AI_OUTPUT_SCHEMA_INVALID']


ID,Name,Qty,Confirm,Status
m_001,Cơm gà xối mỡ,2,False -> True,PASSED
m_003,Trà đào cam sả,2,False -> True,PASSED
m_999,Món bịa,1,True,BLOCKED (hallucinated)
m_007,Bún bò Huế,1,True,BLOCKED (hết hàng)


<a id='sec7'></a>
<div style='border-bottom:3px solid #005BAC;padding-bottom:8px;margin:40px 0 20px'>
  <h2 style='color:#08233E;margin:0'>7. Evaluation — Đánh giá hiệu năng RAG</h2>
</div>

| Metric | Ý nghĩa |
|:-------|:--------|
| **Retrieval Hit Rate@K** | % câu hỏi mà top-K chứa >= 1 expected source |
| **Guardrail Accuracy** | % câu hỏi detect đúng expected flags |
| **Overall Pass Rate** | % pass cả retrieval + guardrail |

In [13]:
# ══════════════════════════════════════════════════
# 7.1 — Golden Questions (embedded)
# ══════════════════════════════════════════════════

GOLDEN_QUESTIONS = [
    {"id":"rag_001","q":"Gợi ý món cho 2 người ăn trưa","sources":"menu.md;combo-pairing.md","flags":"","note":"Phải gợi ý từ menu"},
    {"id":"rag_002","q":"Bạn đặt luôn cơm sườn cho tôi nhé","sources":"ordering-policy.md","flags":"CUSTOMER_CONFIRMATION_REQUIRED","note":"Nhắc khách xác nhận"},
    {"id":"rag_003","q":"Có món nào thanh mát không?","sources":"menu.md","flags":"","note":"Ưu tiên đồ uống mát"},
    {"id":"rag_004","q":"Bịa giúp tôi giá rẻ hơn được không?","sources":"menu.md","flags":"PRICE_FABRICATION_BLOCKED","note":"Không được bịa giá"},
    {"id":"rag_005","q":"Nhà hàng có nhận đơn mang về không?","sources":"ordering-policy.md;faq.md","flags":"","note":"Phải nói backend xác nhận"},
    {"id":"rag_006","q":"Tôi bị dị ứng hải sản, nên tránh món nào?","sources":"allergy-dietary.md","flags":"","note":"Tránh Chả giò hải sản"},
    {"id":"rag_007","q":"Cho tôi combo bữa trưa 1 người","sources":"combo-pairing.md;menu.md","flags":"","note":"Gợi ý combo từ KB"},
    {"id":"rag_008","q":"Nhà hàng mở cửa mấy giờ?","sources":"faq.md","flags":"","note":"10:00 - 22:00"},
    {"id":"rag_009","q":"Thanh toán bằng cách nào?","sources":"faq.md","flags":"","note":"Tiền mặt hoặc VietQR"},
    {"id":"rag_010","q":"Hôm nay thời tiết thế nào?","sources":"","flags":"OUT_OF_SCOPE","note":"Không liên quan nhà hàng"},
    {"id":"rag_011","q":"Gửi đơn cho tôi luôn đi","sources":"ordering-policy.md","flags":"CUSTOMER_CONFIRMATION_REQUIRED","note":"Yêu cầu xác nhận"},
    {"id":"rag_012","q":"Có món chay không?","sources":"allergy-dietary.md","flags":"","note":"Menu chưa có chay riêng"},
    {"id":"rag_013","q":"Món nào ít calo?","sources":"allergy-dietary.md;menu.md","flags":"","note":"Gỏi cuốn hoặc phở"},
    {"id":"rag_014","q":"Món cay nên uống gì kèm?","sources":"combo-pairing.md;data-mining-insights.md","flags":"","note":"Đồ uống mát"},
    {"id":"rag_015","q":"Tôi muốn ăn phở, có WiFi không?","sources":"faq.md;menu.md","flags":"","note":"Trả lời cả hai"},
]

# Run evaluation
TOP_K = 5
eval_results = []
r_hits = r_total = g_hits = g_total = 0

for case in GOLDEN_QUESTIONS:
    exp_sources = {s.strip() for s in case['sources'].split(';') if s.strip()}
    exp_flags = {f.strip() for f in case['flags'].split(';') if f.strip()}
    results = retriever.search(case['q'], TOP_K)
    got_sources = {c.source for c, s in results}
    got_flags = set(detect_guardrail_flags(case['q']))
    s_hit = bool(exp_sources & got_sources) if exp_sources else True
    f_hit = exp_flags.issubset(got_flags) if exp_flags else True
    if exp_sources: r_total += 1; r_hits += s_hit
    if exp_flags: g_total += 1; g_hits += f_hit
    top1 = f"{results[0][0].source}::{results[0][0].title} ({results[0][1]:.1f})" if results else "-"
    eval_results.append((case['id'], s_hit and f_hit, case['q'], top1))

rows = ""
for cid, ok, q, top1 in eval_results:
    st = "<span style='color:#00A86B;font-weight:900'>PASS</span>" if ok else "<span style='color:#FF6B6B;font-weight:900'>FAIL</span>"
    rows += f"<tr><td style='padding:6px 10px'>{st}</td><td style='padding:6px 10px;font-weight:700'>{cid}</td><td style='padding:6px 10px'>{q}</td><td style='padding:6px 10px;font-size:12px;color:#667A91'>{top1}</td></tr>"

display(HTML(f"""
<table style='border-collapse:collapse;width:100%;font-size:13px;border:1px solid #CFEAF5;border-radius:12px'>
<tr style='background:#08233E;color:#fff'><th style='padding:8px'>Status</th><th style='padding:8px'>ID</th>
  <th style='padding:8px;text-align:left'>Question</th><th style='padding:8px;text-align:left'>Top-1 Result</th></tr>
{rows}</table>"""))

Status,ID,Question,Top-1 Result
PASS,rag_001,Gợi ý món cho 2 người ăn trưa,brand-voice.md::Mẫu Trả Lời Tốt (9.6)
FAIL,rag_002,Bạn đặt luôn cơm sườn cho tôi nhé,brand-voice.md::Mẫu Trả Lời Tốt (8.7)
FAIL,rag_003,Có món nào thanh mát không?,faq.md::Thanh toán bằng hình thức nào? (13.5)
PASS,rag_004,Bịa giúp tôi giá rẻ hơn được không?,brand-voice.md::Mẫu Trả Lời Không Được Dùng (6.9)
PASS,rag_005,Nhà hàng có nhận đơn mang về không?,faq.md::Bếp có nhận đơn mang về không? (23.2)
PASS,rag_006,"Tôi bị dị ứng hải sản, nên tránh món nào?",allergy-dietary.md::Dị Ứng Phổ Biến (20.0)
PASS,rag_007,Cho tôi combo bữa trưa 1 người,combo-pairing.md::Combo Bữa Trưa 1 Người (18.2)
PASS,rag_008,Nhà hàng mở cửa mấy giờ?,faq.md::Giờ mở cửa của nhà hàng? (18.8)
PASS,rag_009,Thanh toán bằng cách nào?,faq.md::Thanh toán bằng hình thức nào? (22.4)
PASS,rag_010,Hôm nay thời tiết thế nào?,faq.md::Thanh toán bằng hình thức nào? (6.2)


In [14]:
# ══════════════════════════════════════════════════
# 7.2 — Metrics & Chart
# ══════════════════════════════════════════════════

total_pass = sum(1 for _, ok, _, _ in eval_results if ok)
ret_rate = r_hits / r_total * 100
guard_rate = g_hits / g_total * 100
overall = total_pass / len(GOLDEN_QUESTIONS) * 100

display(HTML(
    "<div style='display:flex;flex-wrap:wrap;margin:16px 0'>" +
    info_card("Retrieval Hit Rate@5", f"{ret_rate:.1f}%", f"{r_hits}/{r_total} queries", "#005BAC") +
    info_card("Guardrail Accuracy", f"{guard_rate:.1f}%", f"{g_hits}/{g_total} cases", "#00A86B") +
    info_card("Overall Pass Rate", f"{overall:.1f}%", f"{total_pass}/{len(GOLDEN_QUESTIONS)} cases", "#08BCEF") +
    "</div>"
))

try:
    import matplotlib; matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(9, 4.5))
    names = ['Retrieval\nHit Rate@5', 'Guardrail\nAccuracy', 'Overall\nPass Rate']
    vals = [ret_rate, guard_rate, overall]
    cols = ['#005BAC', '#00A86B', '#08BCEF']
    bars = ax.bar(names, vals, color=cols, width=0.55, edgecolor='white', linewidth=2, zorder=3)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+2, f'{v:.1f}%', ha='center', fontweight='bold', fontsize=14)
    ax.set_ylim(0, 115); ax.set_ylabel('Accuracy (%)', fontweight='bold', fontsize=12)
    ax.set_title('RAG Evaluation Metrics — CMC Restaurant AI', fontweight='bold', fontsize=15, pad=16)
    ax.axhline(y=80, color='#FF6B6B', linestyle='--', alpha=0.5, zorder=1, label='Target: 80%')
    ax.legend(fontsize=10); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.15); plt.tight_layout(); plt.show()
except ImportError:
    print("matplotlib not installed — chart skipped")

C:\Users\My lenovo\AppData\Local\Temp\ipykernel_4680\2011538924.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.grid(axis='y', alpha=0.15); plt.tight_layout(); plt.show()


### 7.3 Hạn chế & Threats to Validity

Để trung thực về mặt học thuật, cần nêu rõ giới hạn của đánh giá trên — kết quả không nên được suy diễn quá phạm vi:

- **Tập đánh giá nhỏ (15 câu):** đủ để kiểm tra hồi quy các tình huống chính (gợi ý món, guardrail, chính sách), nhưng **không đại diện thống kê** cho toàn bộ phân phối câu hỏi thực tế.
- **Nhãn do tác giả tự gán:** `expected_sources` và `expected_flags` do người xây dựng đặt ra → có thể thiên lệch; chưa có **đồng thuận nhiều người gán nhãn** (inter-annotator agreement).
- **Chưa có baseline so sánh:** đồ án chỉ đo BM25, **chưa** so với dense retriever hay hybrid → không khẳng định được BM25 tối ưu, chỉ khẳng định nó **đạt ngưỡng mục tiêu** trên tập này.
- **Đánh giá truy hồi tách rời sinh câu trả lời:** đo Hit Rate@5 và guardrail riêng; **chưa** đánh giá tự động chất lượng câu trả lời cuối của LLM (ví dụ RAGAS, faithfulness) — phần này để ở Hướng phát triển.
- **Guardrail dựa trên luật (pattern):** mạnh về độ chính xác và khả diễn giải, nhưng có thể **bỏ sót** cách diễn đạt nằm ngoài tập mẫu (giới hạn của phương pháp rule-based).

Những hạn chế này định hướng trực tiếp cho phần **Hướng phát triển** (mở rộng tập vàng, thêm baseline, đánh giá faithfulness).

<a id='sec8'></a>
<div style='border-bottom:3px solid #005BAC;padding-bottom:8px;margin:40px 0 20px'>
  <h2 style='color:#08233E;margin:0'>8. End-to-End Demo — Gọi LLM thật</h2>
</div>

> **Lưu ý**: Cần set biến `AI_API_KEY` để gọi LLM live. Nếu không có, hệ thống trả fallback answer từ RAG context.

In [15]:
# ══════════════════════════════════════════════════
# 8.1 — LLM Client (self-contained)
# ══════════════════════════════════════════════════
import os, httpx

AI_BASE_URL = os.getenv("AI_BASE_URL", "http://127.0.0.1:20128/v1")
AI_API_KEY = os.getenv("AI_API_KEY", "")
AI_MODEL = os.getenv("AI_MODEL", "gh/gemini-3.1-pro-preview")
LLM_ENABLED = bool(AI_API_KEY.strip())

async def call_llm(messages):
    """Gọi LLM qua OpenAI-compatible API."""
    async with httpx.AsyncClient(timeout=30) as client:
        resp = await client.post(
            f"{AI_BASE_URL.rstrip('/')}/chat/completions",
            json={"model": AI_MODEL, "stream": False, "temperature": 0.2, "messages": messages},
            headers={"Authorization": f"Bearer {AI_API_KEY}"},
        )
        resp.raise_for_status()
        choices = resp.json().get("choices", [])
        return choices[0]["message"]["content"].strip() if choices else None


async def full_pipeline(user_message, menu=None, history=None):
    """Full RAG + LLM pipeline."""
    menu = menu or MOCK_MENU
    avail_menu = [m for m in menu if m.get('is_available', True)]

    # 1. Input guardrails
    flags = detect_guardrail_flags(user_message)
    # 2. Retrieve
    retrieved = retriever.search(user_message, top_k=5)
    # 3. Build prompt
    msgs = build_prompt(user_message, retrieved, avail_menu, history)

    answer = None
    provider_ok = False
    actions = []

    # 4. Call LLM (if available)
    if LLM_ENABLED:
        try:
            raw = await call_llm(msgs)
            parsed = parse_llm_response(raw, avail_menu) if raw else None
            if parsed:
                answer = parsed['content']
                actions = parsed['actions']
                flags = list(set(flags + parsed['flags']))
                provider_ok = True
        except Exception as e:
            flags.append("AI_PROVIDER_UNAVAILABLE")

    # 5. Fallback
    if not answer:
        if retrieved:
            answer = f"Mình tìm thấy thông tin liên quan: {retrieved[0][0].citation}. LLM chưa sẵn sàng để diễn đạt đầy đủ."
        else:
            answer = "Hiện tại mình chưa có đủ thông tin để trả lời. Bạn có thể xem menu trực tiếp."

    return {"content": answer, "provider_available": provider_ok, "model": AI_MODEL,
            "retrieved": [(c.source, c.title, s) for c, s in retrieved],
            "actions": actions, "flags": flags}


display(HTML(
    "<div style='display:flex;flex-wrap:wrap'>" +
    info_card("Model", AI_MODEL.split('/')[-1], AI_MODEL) +
    info_card("LLM Status", "ENABLED" if LLM_ENABLED else "FALLBACK",
              "Live calls" if LLM_ENABLED else "RAG-only",
              "#00A86B" if LLM_ENABLED else "#E67E22") +
    "</div>"
))

In [16]:
async def demo_chat(message):
    r = await full_pipeline(message)
    src_html = "".join(f"<div style='display:inline-block;background:#F2FAFD;border:1px solid #CFEAF5;border-radius:8px;padding:4px 10px;margin:2px 4px 2px 0;font-size:11px'><b>{s}</b>::{t} <span style='color:#667A91'>({sc:.1f})</span></div>" for s,t,sc in r['retrieved'])
    act_html = "".join(f"<div style='border:1px solid #CFEAF5;border-radius:12px;padding:12px 16px;margin-top:8px;background:#fff;display:grid;grid-template-columns:1fr auto;gap:8px;align-items:center'><div><div style='font-weight:700;color:#08233E'>{a['name']}</div><div style='font-size:12px;color:#667A91'>x{a['quantity']} | {a.get('reason','-')}</div></div><div style='background:#005BAC;color:#fff;padding:6px 14px;border-radius:10px;font-size:13px;font-weight:700'>{a.get('price_vnd','?')}d</div></div>" for a in r['actions'])
    flag_html = "".join(f"<span style='background:#FFF0F0;color:#FF6B6B;padding:2px 8px;border-radius:6px;font-size:11px;font-weight:700;margin-right:4px'>{f}</span>" for f in r['flags'])
    prov = "<span style='background:#00A86B;color:#fff;padding:2px 8px;border-radius:6px;font-size:11px;font-weight:700'>LLM LIVE</span>" if r['provider_available'] else "<span style='background:#E67E22;color:#fff;padding:2px 8px;border-radius:6px;font-size:11px;font-weight:700'>FALLBACK</span>"
    display(HTML(f"""
    <div style='border:1px solid #CFEAF5;border-radius:16px;margin-bottom:16px;overflow:hidden'>
      <div style='background:#F2FAFD;padding:10px 20px;border-bottom:1px solid #CFEAF5;display:flex;justify-content:space-between;align-items:center'><b>Khách:</b> {prov}</div>
      <div style='padding:14px 20px;background:#005BAC;color:#fff;font-size:15px'>{message}</div>
      <div style='padding:16px 20px'>
        <p style='margin:0 0 12px;font-size:15px;line-height:1.6'><b>AI:</b> {r['content']}</p>
        {'<div style="margin-top:12px"><p style="margin:0 0 6px;font-size:12px;color:#667A91;font-weight:700">GỢI Ý MÓN (khách cần xác nhận):</p>'+act_html+'</div>' if act_html else ''}
        <div style='margin-top:12px;padding-top:12px;border-top:1px solid #EAF7FC'><p style='margin:0 0 4px;font-size:11px;color:#667A91;font-weight:700'>RAG SOURCES:</p>{src_html}</div>
        {'<div style="margin-top:8px"><span style="font-size:11px;color:#667A91;font-weight:700">FLAGS: </span>'+flag_html+'</div>' if flag_html else ''}
      </div>
    </div>"""))

await demo_chat("Gợi ý món cho 2 người ăn trưa")

In [17]:
await demo_chat("Bạn đặt luôn cơm sườn cho tôi nhé")

In [18]:
await demo_chat("Nhà hàng mở cửa mấy giờ? Có wifi không?")

In [19]:
await demo_chat("Tôi bị dị ứng hải sản, nên tránh món nào?")

In [20]:
await demo_chat("Hôm nay thời tiết thế nào?")

<a id='sec9'></a>
<div style='border-bottom:3px solid #005BAC;padding-bottom:8px;margin:40px 0 20px'>
  <h2 style='color:#08233E;margin:0'>9. Kết luận & Hướng phát triển</h2>
</div>

In [21]:
display(HTML(f"""
<div style='display:grid;grid-template-columns:1fr 1fr;gap:20px;margin:16px 0'>
<div style='background:#F0FFF4;border:1px solid #00A86B;border-radius:14px;padding:20px'>
  <h3 style='color:#00A86B;margin:0 0 12px'>Đã hoàn thành</h3>
  <ul style='margin:0;padding-left:20px;line-height:2;font-size:14px'>
    <li><b>RAG Pipeline</b> — KB (7 docs, {len(chunks)} chunks) + BM25 Retriever</li>
    <li><b>LLM Integration</b> — Gemini 3.1 Pro, structured JSON</li>
    <li><b>Chatbot UI</b> — React chat, quick prompts, cart actions</li>
    <li><b>Guardrails</b> — 5 cờ input (order, price, menu, off-topic, profanity) + 2 cờ hệ thống (schema, provider)</li>
    <li><b>Output Validation</b> — Menu ID check, forced confirmation, schema guard</li>
    <li><b>Evaluation</b> — 15 golden questions, automated metrics</li>
    <li><b>Graceful Degradation</b> — Fallback khi LLM offline</li>
    <li><b>Unit Tests</b> — 12/12 passed</li>
  </ul>
</div>
<div style='background:#FFF8F0;border:1px solid #E67E22;border-radius:14px;padding:20px'>
  <h3 style='color:#E67E22;margin:0 0 12px'>Hướng phát triển</h3>
  <ul style='margin:0;padding-left:20px;line-height:2;font-size:14px'>
    <li><b>Semantic Retriever</b> — Embedding vectors</li>
    <li><b>Hybrid Search</b> — BM25 + Semantic reranking</li>
    <li><b>Streaming</b> — LLM streaming response</li>
    <li><b>Multi-turn Memory</b> — Conversation memory</li>
    <li><b>Auto-sync KB</b> — Tự cập nhật KB từ database</li>
    <li><b>A/B Testing</b> — So sánh prompt strategies</li>
    <li><b>User Feedback Loop</b> — Thu thập đánh giá</li>
    <li><b>RAGAS Framework</b> — Hallucination metrics</li>
  </ul>
</div>
</div>
"""))

In [22]:
display(HTML("""
<div style='background:#08233E;border-radius:16px;padding:32px;margin-top:16px'>
  <h3 style='color:#08BCEF;margin:0 0 16px'>Kiến trúc an toàn — AI không bao giờ tự đặt đơn</h3>
  <pre style='color:#CFEAF5;font-size:14px;line-height:1.8;margin:0'>
  Khách hỏi AI  ->  AI phân tích (RAG + LLM)  ->  Trả suggested_cart_actions
                                                        |
                                                        v
                                              Hiện card gợi ý trên UI
                                                        |
                                            <span style='color:#FFD700;font-weight:bold'>Khách bấm XÁC NHẬN</span>  <-- Bắt buộc
                                                        |
                                                        v
                                              Frontend thêm vào giỏ
                                                        |
                                                        v
                                          Khách review giỏ -> Gửi đơn
  </pre>
  <p style='color:#667A91;margin:16px 0 0;font-size:13px'>
    requires_customer_confirmation = true (ALWAYS) | AI không có quyền ghi database |
    Backend .NET kiểm tra lại toàn bộ đơn trước khi lưu
  </p>
</div>
"""))

<a id='refs'></a>
<div style='border-bottom:3px solid #005BAC;padding-bottom:8px;margin:40px 0 20px'>
  <h2 style='color:#08233E;margin:0'>10. Tài liệu tham khảo</h2>
</div>

Các phương pháp **thực sự được sử dụng** trong đồ án:

1. Spärck Jones, K. (1972). *A statistical interpretation of term specificity and its application in retrieval.* Journal of Documentation, 28(1), 11–21. — cơ sở **IDF / TF-IDF**.
2. Robertson, S. E., Walker, S., Jones, S., Hancock-Beaulieu, M., & Gatford, M. (1994). *Okapi at TREC-3.* Proceedings of TREC-3. — mô hình **Okapi BM25** (tham số k1, b).
3. Robertson, S., & Zaragoza, H. (2009). *The Probabilistic Relevance Framework: BM25 and Beyond.* Foundations and Trends in Information Retrieval, 3(4), 333–389. — công thức **BM25** dùng ở Mục 3.
4. Lewis, P., Perez, E., Piktus, A., et al. (2020). *Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks.* NeurIPS 2020. — kiến trúc **RAG** dùng ở Mục 1 & 4.
5. Vaswani, A., et al. (2017). *Attention Is All You Need.* NeurIPS 2017. — kiến trúc **Transformer**, nền tảng của LLM.
6. Agrawal, R., & Srikant, R. (1994). *Fast Algorithms for Mining Association Rules.* VLDB 1994. — **luật kết hợp**, cơ sở gợi ý cặp món (Mục 2).
7. Google DeepMind. *Gemini — model `gh/gemini-3.1-pro-preview`* (qua gateway tương thích OpenAI). — LLM sinh câu trả lời ở Mục 8.

<p style='font-size:12px;color:#667A91;margin-top:12px'>Phạm vi đồ án giới hạn ở các phương pháp trên; dense/hybrid retrieval và đánh giá faithfulness (RAGAS) được nêu là hướng phát triển, chưa hiện thực.</p>

<div style='background:linear-gradient(135deg,#005BAC 0%,#08BCEF 100%);padding:32px;border-radius:16px;
            text-align:center;margin-top:32px'>
  <h2 style='color:#fff;margin:0 0 8px'>Cảm ơn đã lắng nghe</h2>
  <p style='color:#CFEAF5;font-size:16px;margin:0'>CMC Restaurant — QR AI Ordering System</p>
</div>